# Pure-State Entanglement Vs System Size

This notebook is a local CPU report for existing pure-state strip-entanglement products. It reads saved derived observables, validates the campaign, builds flattened-H-style `S/ln2` tables and fits, and displays figures in analysis order. Different system sizes are not mixed in the same plot; within a fixed `N16xNy` system, the available hyperparameters are grouped as subplots/curves. It does not run circuit dynamics or recompute covariance eigensystems.


## Configurations And Domain-Wall Geometry

The pure-state analysis reads the existing strip-entropy products derived from covariance snapshots. The four source covariance runs are:

| Nx | Ny | nshell | protocol | samples | cycles | saved snapshots | init state | dtype | dw_truncation | alpha_1 / alpha_2 |
|---:|---:|---:|---|---:|---:|---|---|---|---|---|
| 16 | 30 | 1 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default random pure state | complex128 | True | 1 / 30 |
| 16 | 30 | 2 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default random pure state | complex128 | True | 1 / 30 |
| 16 | 40 | 1 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default random pure state | complex128 | True | 1 / 30 |
| 16 | 40 | 2 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default random pure state | complex128 | True | 1 / 30 |

Domain-wall convention for all rows: `Nx=16` gives `half=Nx//2=8` and `w=floor(0.2*Nx)=3`, so the code stores `DW_loc=[5, 11]`. The topological `alpha_1` slab is `x=5..11` inclusive, the trivial `alpha_2` regions are `x=0..4` and `x=12..15`, and the two domain-wall locations are reported at `x=5` and `x=11` in the simulation metadata/convention. These walls extend along the full periodic `y` direction for both `Ny=30` and `Ny=40`.

Entropy dataframes, fit slopes, and rendered figures use `S/ln2` units. The source `.npz` products are validated in their stored natural-log units before normalization.


## 1. Setup

Locate the small-system bundle, define the expected pure-state strip campaign, and choose the fit/window conventions used below.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BUNDLE_NAME = 'colab_small_system_testing'
candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'gpu_data' / 'pure_state_strip_entropy_contours').exists():
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME}/gpu_data/pure_state_strip_entropy_contours')

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
SOURCE_MANIFEST_PATH = GPU_DATA_ROOT / 'pure_state_strip_entropy_contours' / 'campaign_manifest.json'
OUTPUT_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'pure_state_entanglement_vs_system_size_cpu'
FIG_ROOT = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIGS = {(16, 30, 1), (16, 30, 2), (16, 40, 1), (16, 40, 2)}
EXPECTED_SNAPSHOT_CYCLES = np.asarray([5, 10, 20, 50], dtype=np.int64)
PRIMARY_CYCLE = 50
FIT_AY_MIN = 2
EPS_NEGATIVE = 1e-10
TOTAL_TOL = 1e-8
ENTROPY_NORM = float(np.log(2.0))
ENTROPY_UNITS = 'S_over_ln2'

X_INTERVALS = {
    'full_x': np.arange(0, 16, dtype=np.int64),
    'left_wall_window': np.arange(4, 7, dtype=np.int64),
    'right_wall_window': np.arange(10, 13, dtype=np.int64),
}
TARGET_SLOPES_NATURAL_LOG = {'full_x': 1.0 / 3.0, 'left_wall_window': 1.0 / 6.0, 'right_wall_window': 1.0 / 6.0}
TARGET_SLOPES = {key: value / ENTROPY_NORM for key, value in TARGET_SLOPES_NATURAL_LOG.items()}

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] source manifest: {SOURCE_MANIFEST_PATH}')
print(f'[setup] output root: {OUTPUT_ROOT}')


## 2. Shared Helpers

Small utilities for JSON/NPZ output, finite-size fit coordinates, and inline display of saved figures.

In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp.replace(path)


def save_npz_atomic(path: Path, **arrays) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp.replace(path)


def rel_to_bundle(path: Path) -> str:
    return str(Path(path).resolve().relative_to(BUNDLE_ROOT))


def fit_line_with_error(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3:
        return {'slope': np.nan, 'slope_err': np.nan, 'intercept': np.nan, 'r2': np.nan, 'x': x, 'y': y, 'y_fit': np.full_like(y, np.nan)}
    try:
        coeffs, cov = np.polyfit(x, y, 1, cov=True)
        slope_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan
    except Exception:
        coeffs = np.polyfit(x, y, 1)
        slope_err = np.nan
    slope, intercept = float(coeffs[0]), float(coeffs[1])
    y_fit = slope * x + intercept
    ss_res = float(np.sum((y - y_fit) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    return {'slope': slope, 'slope_err': slope_err, 'intercept': intercept, 'r2': float(r2), 'x': x, 'y': y, 'y_fit': y_fit}


def chord_length(Ay, Ny):
    Ay = np.asarray(Ay, dtype=np.float64)
    return (float(Ny) / np.pi) * np.sin(np.pi * Ay / float(Ny))


def log_sin_chord(Ay, Ny):
    Ay = np.asarray(Ay, dtype=np.float64)
    out = np.full(Ay.shape, np.nan, dtype=np.float64)
    mask = Ay > 0
    out[mask] = np.log(np.sin(np.pi * Ay[mask] / float(Ny)))
    return out


def case_id_from_summary(summary: dict) -> str:
    return f"N{summary['Nx']}x{summary['Ny']}_nsh{summary['nshell']}_{summary['protocol']}"



def show_saved_figure(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Image(filename=str(path)))


def system_label(nx: int, ny: int) -> str:
    return f'N{int(nx)}x{int(ny)}'


def outside_legend(ax, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0.0,
            frameon=True,
            **kwargs,
        )


def remove_stale_figures(names: list[str]) -> None:
    for name in names:
        path = FIG_ROOT / name
        if path.exists():
            path.unlink()
            print(f'[cleanup] removed stale figure {path.name}')


## 3. Load And Validate The Source Campaign

The source is `gpu_data/pure_state_strip_entropy_contours/campaign_manifest.json`. Validation checks the four expected `N16x{30,40}` and `nshell={1,2}` pure-state runs, all saved `Ay` products, contour shapes, finiteness, and total entropy consistency.

In [ ]:
REQUIRED_KEYS = {
    'entropy_contour_avg', 'entropy_contour_std', 'entropy_contour_sem',
    'entropy_total_mean', 'entropy_total_std', 'entropy_total_sem',
    'snapshot_cycles', 'x', 'strip_dy', 'Ay', 'sample_count', 'y0_count',
    'observation_count', 'config_json',
}


def validate_source_manifest(manifest: dict) -> None:
    results = manifest.get('results', [])
    if len(results) != 4:
        raise ValueError(f'Expected 4 pure-state strip runs, got {len(results)}')
    observed = {(int(r['Nx']), int(r['Ny']), int(r['nshell'])) for r in results}
    if observed != EXPECTED_CONFIGS:
        raise ValueError(f'Unexpected configs: {sorted(observed)}')
    for r in results:
        if r.get('protocol') != 'perfect_correction':
            raise ValueError(f"Unexpected protocol {r.get('protocol')}")
        if list(map(int, r.get('snapshot_cycles', []))) != EXPECTED_SNAPSHOT_CYCLES.tolist():
            raise ValueError(f"Unexpected snapshot cycles for {r}")


def load_run_products(result: dict) -> tuple[dict, dict, list[dict]]:
    summary_path = GPU_DATA_ROOT / result['summary_path_relative']
    summary = load_json(summary_path)
    case_id = case_id_from_summary(summary)
    Nx, Ny, nshell = int(summary['Nx']), int(summary['Ny']), int(summary['nshell'])
    if (Nx, Ny, nshell) not in EXPECTED_CONFIGS:
        raise ValueError(f'{case_id}: unexpected config')
    if summary.get('protocol') != 'perfect_correction':
        raise ValueError(f'{case_id}: unexpected protocol={summary.get("protocol")}')
    if np.asarray(summary['snapshot_cycles'], dtype=np.int64).tolist() != EXPECTED_SNAPSHOT_CYCLES.tolist():
        raise ValueError(f'{case_id}: unexpected snapshot cycles')
    expected_Ay = list(range(0, Ny // 2 + 1))
    products = sorted(summary.get('products', []), key=lambda item: int(item['Ay']))
    if [int(item['Ay']) for item in products] != expected_Ay:
        raise ValueError(f'{case_id}: expected Ay {expected_Ay}, got {[int(item["Ay"]) for item in products]}')

    validation = {
        'case_id': case_id,
        'summary_path': rel_to_bundle(summary_path),
        'product_count': len(products),
        'max_total_mismatch': 0.0,
        'min_contour_value': np.inf,
        'max_contour_value': -np.inf,
        'Ay_zero_ok': False,
    }
    loaded = []
    for product in products:
        product_path = GPU_DATA_ROOT / product['product_path_relative']
        if not product_path.exists():
            raise FileNotFoundError(product_path)
        with np.load(product_path, allow_pickle=False) as data:
            missing = sorted(REQUIRED_KEYS - set(data.files))
            if missing:
                raise KeyError(f'{product_path} missing keys {missing}')
            Ay = int(np.asarray(data['Ay']).item())
            if Ay != int(product['Ay']):
                raise ValueError(f'{product_path}: product Ay mismatch')
            cycles = np.asarray(data['snapshot_cycles'], dtype=np.int64)
            contour = np.asarray(data['entropy_contour_avg'], dtype=np.float64)
            contour_std = np.asarray(data['entropy_contour_std'], dtype=np.float64)
            contour_sem = np.asarray(data['entropy_contour_sem'], dtype=np.float64)
            total = np.asarray(data['entropy_total_mean'], dtype=np.float64)
            total_std = np.asarray(data['entropy_total_std'], dtype=np.float64)
            total_sem = np.asarray(data['entropy_total_sem'], dtype=np.float64)
            if not np.array_equal(cycles, EXPECTED_SNAPSHOT_CYCLES):
                raise ValueError(f'{product_path}: unexpected snapshot cycles {cycles.tolist()}')
            expected_shape = (len(EXPECTED_SNAPSHOT_CYCLES), Nx, Ay)
            if contour.shape != expected_shape:
                raise ValueError(f'{product_path}: expected contour shape {expected_shape}, got {contour.shape}')
            if total.shape != (len(EXPECTED_SNAPSHOT_CYCLES),):
                raise ValueError(f'{product_path}: expected total shape {(len(EXPECTED_SNAPSHOT_CYCLES),)}, got {total.shape}')
            numeric = [contour, contour_std, contour_sem, total, total_std, total_sem]
            if not all(np.all(np.isfinite(arr)) for arr in numeric):
                raise FloatingPointError(f'{product_path}: non-finite numeric values')
            contour_min = float(np.min(contour)) if contour.size else 0.0
            contour_max = float(np.max(contour)) if contour.size else 0.0
            if contour_min < -EPS_NEGATIVE:
                raise ValueError(f'{product_path}: negative contour minimum {contour_min}')
            total_from_contour = contour.sum(axis=(1, 2))
            total_mismatch = float(np.max(np.abs(total_from_contour - total)))
            if total_mismatch > TOTAL_TOL:
                raise ValueError(f'{product_path}: total mismatch {total_mismatch:g}')
            if Ay == 0:
                if contour.shape != (len(EXPECTED_SNAPSHOT_CYCLES), Nx, 0) or np.max(np.abs(total)) > TOTAL_TOL:
                    raise ValueError(f'{product_path}: invalid Ay=0 product')
                validation['Ay_zero_ok'] = True
            config = json.loads(str(np.asarray(data['config_json']).item()))
            if int(config['Nx']) != Nx or int(config['Ny']) != Ny or int(config['nshell']) != nshell:
                raise ValueError(f'{product_path}: config_json mismatch')
            validation['max_total_mismatch'] = max(validation['max_total_mismatch'], total_mismatch)
            validation['min_contour_value'] = min(validation['min_contour_value'], contour_min)
            validation['max_contour_value'] = max(validation['max_contour_value'], contour_max)
            loaded.append({
                'Ay': Ay,
                'product_path': product_path,
                'snapshot_cycles': cycles,
                'contour': contour,
                'contour_std': contour_std,
                'contour_sem': contour_sem,
                'total': total,
                'total_std': total_std,
                'total_sem': total_sem,
                'sample_count': int(np.asarray(data['sample_count']).item()),
                'y0_count': int(np.asarray(data['y0_count']).item()),
                'observation_count': int(np.asarray(data['observation_count']).item()),
            })
    if not validation['Ay_zero_ok']:
        raise ValueError(f'{case_id}: missing valid Ay=0 product')
    return summary, validation, loaded


source_manifest = load_json(SOURCE_MANIFEST_PATH)
validate_source_manifest(source_manifest)
run_payloads = []
validation_report = {
    'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
    'expected_snapshot_cycles': EXPECTED_SNAPSHOT_CYCLES.tolist(),
    'runs': {},
}
for result in sorted(source_manifest['results'], key=lambda r: (int(r['nshell']), int(r['Ny']))):
    summary, validation, products = load_run_products(result)
    case_id = case_id_from_summary(summary)
    validation_report['runs'][case_id] = validation
    run_payloads.append({'summary': summary, 'case_id': case_id, 'products': products})

write_json_atomic(OUTPUT_ROOT / 'validation_report.json', validation_report)
print(f'[data] loaded {len(run_payloads)} runs and wrote validation report')


## 4. Build Tables And Log-Chord Fits

This section turns the stored `Ay` products into tabular strip entropy curves, integrated contour curves, and log-chord fit summaries. The contour products are already averaged over samples and over strip origin `y0`; the notebook sums the stored local contour over `(x, dy)` only. For the full strip this is `[0,Nx) x [0,Ay)`, and for wall-window diagnostics it is the selected `x` window times `[0,Ay)`.

The log-chord fit window is the observed half-branch `Ay=2..Ny//2`; `Ay=0` and `Ay=1` are excluded from the fits.

In [ ]:
run_rows = []
strip_rows = []
integrated_rows = []
fit_rows = []

for run in run_payloads:
    summary = run['summary']
    case_id = run['case_id']
    Nx, Ny, nshell = int(summary['Nx']), int(summary['Ny']), int(summary['nshell'])
    run_rows.append({
        'case_id': case_id,
        'Nx': Nx,
        'Ny': Ny,
        'nshell': nshell,
        'protocol': summary['protocol'],
        'samples': int(summary['samples']),
        'snapshot_cycles': json.dumps(summary['snapshot_cycles']),
        'Ay_min': min(int(p['Ay']) for p in run['products']),
        'Ay_max': max(int(p['Ay']) for p in run['products']),
        'source_summary_path': validation_report['runs'][case_id]['summary_path'],
    })

    curve_by_interval_cycle = {}
    for product in run['products']:
        Ay = int(product['Ay'])
        chord = float(chord_length(Ay, Ny)) if Ay > 0 else 0.0
        log_sin = float(log_sin_chord(Ay, Ny)) if Ay > 0 else np.nan
        for t_idx, cycle in enumerate(product['snapshot_cycles']):
            cycle = int(cycle)
            total_mean = float(product['total'][t_idx] / ENTROPY_NORM)
            strip_rows.append({
                'case_id': case_id,
                'Nx': Nx,
                'Ny': Ny,
                'nshell': nshell,
                'cycle': cycle,
                'Ay': Ay,
                'chord_length': chord,
                'log_sin_pi_Ay_over_Ny': log_sin,
                'entropy_mean_over_ln2': total_mean,
                'entropy_std_over_ln2': float(product['total_std'][t_idx] / ENTROPY_NORM),
                'entropy_sem_over_ln2': float(product['total_sem'][t_idx] / ENTROPY_NORM),
                'entropy_units': ENTROPY_UNITS,
                'sample_count': product['sample_count'],
                'y0_count': product['y0_count'],
                'observation_count': product['observation_count'],
            })
            for interval_name, x_values in X_INTERVALS.items():
                x_values = np.asarray(x_values, dtype=np.int64)
                if Ay == 0:
                    integrated_mean = 0.0
                else:
                    integrated_mean = float(product['contour'][t_idx][x_values, :].sum() / ENTROPY_NORM)
                if interval_name == 'full_x':
                    integrated_std = float(product['total_std'][t_idx] / ENTROPY_NORM)
                    integrated_sem = float(product['total_sem'][t_idx] / ENTROPY_NORM)
                    note = 'stored_full_strip_uncertainty'
                else:
                    integrated_std = np.nan
                    integrated_sem = np.nan
                    note = 'mean_only_no_cross_bin_covariance'
                integrated_rows.append({
                    'case_id': case_id,
                    'Nx': Nx,
                    'Ny': Ny,
                    'nshell': nshell,
                    'cycle': cycle,
                    'Ay': Ay,
                    'x_interval_kind': interval_name,
                    'x_interval_start': int(x_values[0]),
                    'x_interval_stop': int(x_values[-1]),
                    'x_interval_size': int(x_values.size),
                    'chord_length': chord,
                    'log_sin_pi_Ay_over_Ny': log_sin,
                    'integrated_entropy_mean_over_ln2': integrated_mean,
                    'integrated_entropy_std_over_ln2': integrated_std,
                    'integrated_entropy_sem_over_ln2': integrated_sem,
                    'entropy_units': ENTROPY_UNITS,
                    'uncertainty_note': note,
                })
                curve_by_interval_cycle.setdefault((cycle, interval_name), []).append((Ay, log_sin, integrated_mean))

    for (cycle, interval_name), points in sorted(curve_by_interval_cycle.items()):
        points = sorted(points, key=lambda item: item[0])
        fit_points = [(Ay, x, y) for Ay, x, y in points if Ay >= FIT_AY_MIN and np.isfinite(x)]
        Ay_fit = np.asarray([p[0] for p in fit_points], dtype=np.int64)
        x_fit = np.asarray([p[1] for p in fit_points], dtype=np.float64)
        y_fit = np.asarray([p[2] for p in fit_points], dtype=np.float64)
        fit = fit_line_with_error(x_fit, y_fit)
        target = TARGET_SLOPES[interval_name]
        slope = float(fit['slope'])
        percent_diff = 100.0 * abs(slope - target) / abs(target) if np.isfinite(slope) else np.nan
        fit_rows.append({
            'percent_diff': percent_diff,
            'slope_over_ln2': slope,
            'target_slope_over_ln2': target,
            'slope_err_over_ln2': float(fit['slope_err']),
            'intercept_over_ln2': float(fit['intercept']),
            'r2': float(fit['r2']),
            'n_fit_points': int(Ay_fit.size),
            'Ay_fit_min': int(Ay_fit.min()) if Ay_fit.size else np.nan,
            'Ay_fit_max': int(Ay_fit.max()) if Ay_fit.size else np.nan,
            'branch': 'observed_half',
            'case_id': case_id,
            'Nx': Nx,
            'Ny': Ny,
            'nshell': nshell,
            'cycle': int(cycle),
            'x_interval_kind': interval_name,
            'x_interval_start': int(X_INTERVALS[interval_name][0]),
            'x_interval_stop': int(X_INTERVALS[interval_name][-1]),
        })

run_index = pd.DataFrame(run_rows).sort_values(['nshell', 'Ny']).reset_index(drop=True)
strip_entropy_curves = pd.DataFrame(strip_rows).sort_values(['nshell', 'Ny', 'cycle', 'Ay']).reset_index(drop=True)
integrated_contour_curves = pd.DataFrame(integrated_rows).sort_values(['nshell', 'Ny', 'cycle', 'x_interval_kind', 'Ay']).reset_index(drop=True)
log_chord_fit_rows = pd.DataFrame(fit_rows).sort_values(['percent_diff', 'cycle', 'x_interval_kind', 'nshell', 'Ny'], na_position='last').reset_index(drop=True)
log_chord_fit_summary = (
    log_chord_fit_rows
    .groupby(['cycle', 'nshell', 'x_interval_kind'], dropna=False)
    .agg(
        mean_percent_diff=('percent_diff', 'mean'),
        max_percent_diff=('percent_diff', 'max'),
        mean_slope_over_ln2=('slope_over_ln2', 'mean'),
        mean_r2=('r2', 'mean'),
        n_fits=('slope_over_ln2', 'count'),
    )
    .reset_index()
    .sort_values(['cycle', 'mean_percent_diff', 'x_interval_kind'])
)

run_index.to_csv(OUTPUT_ROOT / 'run_index.csv', index=False)
strip_entropy_curves.to_csv(OUTPUT_ROOT / 'strip_entropy_curves.csv', index=False)
integrated_contour_curves.to_csv(OUTPUT_ROOT / 'integrated_contour_curves.csv', index=False)
log_chord_fit_rows.to_csv(OUTPUT_ROOT / 'log_chord_fit_rows.csv', index=False)
log_chord_fit_summary.to_csv(OUTPUT_ROOT / 'log_chord_fit_summary.csv', index=False)
print('[tables] wrote CSV outputs')
log_chord_fit_summary.head()


display(Markdown('### Run index'))
display(run_index)
display(Markdown('### Log-chord fit summary (`S/ln2`)'))
display(log_chord_fit_summary)


## 5. Save Reusable Arrays

The padded arrays below make later plotting and checks cheap without reopening each `Ay` product.

In [ ]:
case_ids = [run['case_id'] for run in run_payloads]
max_Ay = max(max(int(p['Ay']) for p in run['products']) for run in run_payloads)
Ay_grid = np.arange(max_Ay + 1, dtype=np.int64)
strip_mean = np.full((len(run_payloads), len(EXPECTED_SNAPSHOT_CYCLES), max_Ay + 1), np.nan, dtype=np.float64)
strip_sem = np.full_like(strip_mean, np.nan)
half_contours_cycle50 = np.full((len(run_payloads), 16, max_Ay), np.nan, dtype=np.float64)
half_Ay = np.zeros((len(run_payloads),), dtype=np.int64)
for run_idx, run in enumerate(run_payloads):
    cycle_index = int(np.where(EXPECTED_SNAPSHOT_CYCLES == PRIMARY_CYCLE)[0][0])
    for product in run['products']:
        Ay = int(product['Ay'])
        strip_mean[run_idx, :, Ay] = product['total'] / ENTROPY_NORM
        strip_sem[run_idx, :, Ay] = product['total_sem'] / ENTROPY_NORM
        if Ay == int(run['summary']['Ny']) // 2 and Ay > 0:
            half_Ay[run_idx] = Ay
            half_contours_cycle50[run_idx, :, :Ay] = product['contour'][cycle_index] / ENTROPY_NORM

save_npz_atomic(
    OUTPUT_ROOT / 'derived_arrays.npz',
    case_id=np.asarray(case_ids),
    snapshot_cycles=EXPECTED_SNAPSHOT_CYCLES,
    Ay_grid=Ay_grid,
    strip_entropy_mean_over_ln2_padded=strip_mean,
    strip_entropy_sem_over_ln2_padded=strip_sem,
    entropy_units=np.asarray(ENTROPY_UNITS),
    half_Ay=half_Ay,
    half_strip_contour_cycle50_padded=half_contours_cycle50,
    config_json=np.asarray(json.dumps({
        'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
        'primary_cycle': PRIMARY_CYCLE,
        'x_intervals': {k: v.astype(int).tolist() for k, v in X_INTERVALS.items()},
        'target_slopes_over_ln2': TARGET_SLOPES,
        'target_slopes_natural_log': TARGET_SLOPES_NATURAL_LOG,
        'entropy_units': ENTROPY_UNITS,
    }, sort_keys=True)),
)
print('[arrays] wrote derived_arrays.npz')


## 6. Strip Entropy Curves

First inspect the raw subsystem-size dependence. The all-cycle panel checks time evolution; the cycle-50 panels are the steady-snapshot comparison used for finite-size diagnostics.

In [ ]:
def line_style_for_cycle(cycle):
    markers = {5: 'o', 10: 's', 20: '^', 50: 'D'}
    linestyles = {5: '-', 10: '--', 20: '-.', 50: ':'}
    return markers.get(int(cycle), 'o'), linestyles.get(int(cycle), '-')

remove_stale_figures([
    'strip_entropy_vs_Ay_by_cycle.png',
    'strip_entropy_vs_Ay_cycle50.png',
    'strip_entropy_vs_chord_cycle50.png',
])

for Ny in sorted(strip_entropy_curves['Ny'].unique()):
    display(Markdown(f'### Strip entropy curves: N16x{int(Ny)}'))
    nshell_values = sorted(strip_entropy_curves[strip_entropy_curves['Ny'] == Ny]['nshell'].unique())

    fig, axes = plt.subplots(1, len(nshell_values), figsize=(7.4 * len(nshell_values), 4.5), constrained_layout=True, squeeze=False)
    axes = axes.ravel()
    for ax, nshell in zip(axes, nshell_values):
        for cycle in EXPECTED_SNAPSHOT_CYCLES:
            sub = strip_entropy_curves[(strip_entropy_curves['nshell'] == nshell) & (strip_entropy_curves['Ny'] == Ny) & (strip_entropy_curves['cycle'] == int(cycle))]
            marker, ls = line_style_for_cycle(cycle)
            ax.plot(sub['Ay'], sub['entropy_mean_over_ln2'], marker=marker, ms=3, lw=1.3, ls=ls, label=f'cycle={int(cycle)}')
        ax.set_title(f'N16x{int(Ny)}, nshell={int(nshell)}')
        ax.set_xlabel('Ay')
        ax.set_ylabel(r'$S(A_y)/\ln 2$')
        outside_legend(ax, fontsize=8)
    path = FIG_ROOT / f'strip_entropy_vs_Ay_N16x{int(Ny)}_by_cycle.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Strip entropy across saved snapshots, N16x{int(Ny)}')

    fig, axes = plt.subplots(1, len(nshell_values), figsize=(7.4 * len(nshell_values), 4.5), constrained_layout=True, squeeze=False, sharey=True)
    axes = axes.ravel()
    for ax, nshell in zip(axes, nshell_values):
        sub = strip_entropy_curves[(strip_entropy_curves['nshell'] == nshell) & (strip_entropy_curves['Ny'] == Ny) & (strip_entropy_curves['cycle'] == PRIMARY_CYCLE)]
        ax.errorbar(sub['Ay'], sub['entropy_mean_over_ln2'], yerr=sub['entropy_sem_over_ln2'], marker='o', ms=3, lw=1.3, capsize=2, label=f'cycle={PRIMARY_CYCLE}')
        ax.set_title(f'N16x{int(Ny)}, nshell={int(nshell)}')
        ax.set_xlabel('Ay')
        ax.set_ylabel(r'$S(A_y)/\ln 2$')
        outside_legend(ax, fontsize=8)
    path = FIG_ROOT / f'strip_entropy_vs_Ay_N16x{int(Ny)}_cycle{PRIMARY_CYCLE}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Steady snapshot strip entropy, N16x{int(Ny)}, cycle {PRIMARY_CYCLE}')

    fig, axes = plt.subplots(1, len(nshell_values), figsize=(7.4 * len(nshell_values), 4.5), constrained_layout=True, squeeze=False, sharey=True)
    axes = axes.ravel()
    for ax, nshell in zip(axes, nshell_values):
        sub = strip_entropy_curves[(strip_entropy_curves['nshell'] == nshell) & (strip_entropy_curves['Ny'] == Ny) & (strip_entropy_curves['cycle'] == PRIMARY_CYCLE) & (strip_entropy_curves['Ay'] > 0)]
        ax.plot(sub['chord_length'], sub['entropy_mean_over_ln2'], marker='o', ms=3, lw=1.3, label=f'cycle={PRIMARY_CYCLE}')
        ax.set_title(f'N16x{int(Ny)}, nshell={int(nshell)}')
        ax.set_xlabel(r'$(N_y/\pi)\sin(\pi A_y/N_y)$')
        ax.set_ylabel(r'$S(A_y)/\ln 2$')
        outside_legend(ax, fontsize=8)
    path = FIG_ROOT / f'strip_entropy_vs_chord_N16x{int(Ny)}_cycle{PRIMARY_CYCLE}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Steady snapshot entropy against chord length, N16x{int(Ny)}, cycle {PRIMARY_CYCLE}')


## 7. Log-Chord Scaling Diagnostics

Integrate the already-`y0`-averaged local contour over the full `x` direction and over the two wall windows, then fit against `log(sin(pi Ay / Ny))`. The fit window is `Ay=2..Ny//2` for each circumference, so the largest fitted `Ay` is 15 for `Ny=30` and 20 for `Ny=40`. The figure legends report the fitted slope `m` next to the target slope for the corresponding interval.

In [ ]:
remove_stale_figures([
    'log_chord_fits_cycle50.png',
    'slope_percent_error_summary_cycle50.png',
])

interval_order = ['full_x', 'left_wall_window', 'right_wall_window']
for Ny in sorted(integrated_contour_curves['Ny'].unique()):
    display(Markdown(f'### Log-chord scaling diagnostics: N16x{int(Ny)}'))
    nshell_values = sorted(integrated_contour_curves[integrated_contour_curves['Ny'] == Ny]['nshell'].unique())
    fig, axes = plt.subplots(len(interval_order), len(nshell_values), figsize=(7.4 * len(nshell_values), 3.7 * len(interval_order)), constrained_layout=True, squeeze=False)
    for r, interval in enumerate(interval_order):
        for c, nshell in enumerate(nshell_values):
            ax = axes[r, c]
            sub = integrated_contour_curves[
                (integrated_contour_curves['nshell'] == nshell)
                & (integrated_contour_curves['Ny'] == Ny)
                & (integrated_contour_curves['cycle'] == PRIMARY_CYCLE)
                & (integrated_contour_curves['x_interval_kind'] == interval)
                & (integrated_contour_curves['Ay'] >= FIT_AY_MIN)
            ]
            fit = log_chord_fit_rows[
                (log_chord_fit_rows['nshell'] == nshell)
                & (log_chord_fit_rows['Ny'] == Ny)
                & (log_chord_fit_rows['cycle'] == PRIMARY_CYCLE)
                & (log_chord_fit_rows['x_interval_kind'] == interval)
            ].iloc[0]
            target = TARGET_SLOPES[interval]
            fit_label = (
                f'm={fit["slope_over_ln2"]:.3f}±{fit["slope_err_over_ln2"]:.3f}; '
                f'target={target:.3f}; R²={fit["r2"]:.3f}; '
                f'Ay={int(fit["Ay_fit_min"])}..{int(fit["Ay_fit_max"])}'
            )
            xfit = np.asarray(sub['log_sin_pi_Ay_over_Ny'], dtype=np.float64)
            ax.plot(xfit, sub['integrated_entropy_mean_over_ln2'], marker='o', ms=3, lw=0, label='data')
            ax.plot(xfit, fit['slope_over_ln2'] * xfit + fit['intercept_over_ln2'], color='black', ls='--', lw=1.3, label=fit_label)
            ax.set_title(f'{interval}, N16x{int(Ny)}, nshell={int(nshell)}')
            ax.set_xlabel(r'$\log[\sin(\pi A_y/N_y)]$')
            ax.set_ylabel(r'$\sum_{x\in I_x}\sum_{dy} s_A(x,dy)/\ln 2$')
            outside_legend(ax, fontsize=7)
    path = FIG_ROOT / f'log_chord_fits_N16x{int(Ny)}_cycle{PRIMARY_CYCLE}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Log-chord fits, N16x{int(Ny)}, cycle {PRIMARY_CYCLE}')

    primary_fits = log_chord_fit_rows[(log_chord_fit_rows['cycle'] == PRIMARY_CYCLE) & (log_chord_fit_rows['Ny'] == Ny)].copy()
    primary_fits['label'] = primary_fits.apply(lambda r: f"nsh{int(r['nshell'])}", axis=1)
    fig, axes = plt.subplots(1, len(interval_order), figsize=(5.2 * len(interval_order), 4.2), constrained_layout=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, interval in zip(axes, interval_order):
        sub = primary_fits[primary_fits['x_interval_kind'] == interval].sort_values(['nshell'])
        ax.bar(sub['label'], sub['percent_diff'], color=['#4c78a8', '#f58518'][:len(sub)])
        ax.set_title(interval)
        ax.set_ylabel('slope percent error')
        ax.tick_params(axis='x', rotation=0)
    path = FIG_ROOT / f'slope_percent_error_summary_N16x{int(Ny)}_cycle{PRIMARY_CYCLE}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Fit slope percent error, N16x{int(Ny)}, cycle {PRIMARY_CYCLE}')

print('[figures] wrote system-size-separated fit figures')
display(Markdown(f'### Cycle {PRIMARY_CYCLE} fit rows (`S/ln2`)'))
display(log_chord_fit_rows[log_chord_fit_rows['cycle'] == PRIMARY_CYCLE].sort_values(['Ny', 'x_interval_kind', 'nshell']).reset_index(drop=True))


## 8. Full-`x` Entanglement With Late-Window Fits

This diagnostic uses only the full-`x` strip entropy, meaning the already-`y0`-averaged contour summed over local coordinates `[0,Nx) x [0,Ay)`. The plotted points show the full available entropy curve for every `Ay=0..Ny//2`, while the transparent grey band marks the fit window `Ay=5..Ny//2`. Slopes are still fit in the log-chord coordinate `log(sin(pi Ay / Ny))` and reported in each panel.

In [ ]:
FULL_X_LATE_FIT_AY_MIN = 8
full_x_late_fit_rows = []
remove_stale_figures([
    'full_x_entropy_late_fit_window_cycle50.png',
    'full_x_entropy_late_fit_window_cycle20.png',
    'full_x_log_chord_window_stability_sweep_cycle50.png',
])
stale_table = OUTPUT_ROOT / 'full_x_log_chord_window_stability_sweep.csv'
if stale_table.exists():
    stale_table.unlink()
    print(f'[cleanup] removed stale table {stale_table.name}')

for Ny in sorted(strip_entropy_curves['Ny'].unique()):
    nshell_values = sorted(strip_entropy_curves[strip_entropy_curves['Ny'] == Ny]['nshell'].unique())
    display(Markdown(f'### Full-x late-window log-chord fits: N16x{int(Ny)}'))
    fig, axes = plt.subplots(1, len(nshell_values), figsize=(7.4 * len(nshell_values), 4.6), constrained_layout=True, squeeze=False, sharey=True)
    axes = axes.ravel()

    for ax, nshell in zip(axes, nshell_values):
        sub = strip_entropy_curves[
            (strip_entropy_curves['nshell'] == nshell)
            & (strip_entropy_curves['Ny'] == Ny)
            & (strip_entropy_curves['cycle'] == PRIMARY_CYCLE)
            & (strip_entropy_curves['Ay'] > 0)
        ].sort_values('Ay')

        if sub.empty:
            raise ValueError(f'Missing full_x entropy curve for nshell={nshell}, Ny={Ny}')

        fit_ay_max = int(Ny) // 2
        fit_mask = (sub['Ay'] >= FULL_X_LATE_FIT_AY_MIN) & (sub['Ay'] <= fit_ay_max)
        fit_sub = sub[fit_mask].copy()

        if len(fit_sub) < 3:
            raise ValueError(f'Not enough fit points for nshell={nshell}, Ny={Ny}: {len(fit_sub)}')

        x_all = sub['log_sin_pi_Ay_over_Ny'].to_numpy(dtype=np.float64)
        y_all = sub['entropy_mean_over_ln2'].to_numpy(dtype=np.float64)
        yerr_all = sub['entropy_sem_over_ln2'].to_numpy(dtype=np.float64)
        x_fit = fit_sub['log_sin_pi_Ay_over_Ny'].to_numpy(dtype=np.float64)
        y_fit_data = fit_sub['entropy_mean_over_ln2'].to_numpy(dtype=np.float64)

        fit = fit_line_with_error(x_fit, y_fit_data)
        target = TARGET_SLOPES['full_x']
        slope = float(fit['slope'])
        intercept = float(fit['intercept'])
        percent_diff = 100.0 * abs(slope - target) / abs(target) if np.isfinite(slope) else np.nan
        x_band_min = float(np.min(x_fit))
        x_band_max = float(np.max(x_fit))
        x_line = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300)
        y_line = slope * x_line + intercept

        full_x_late_fit_rows.append({
            'case_id': f'N16x{int(Ny)}_nsh{int(nshell)}_perfect_correction',
            'Nx': 16,
            'Ny': int(Ny),
            'nshell': int(nshell),
            'cycle': int(PRIMARY_CYCLE),
            'x_interval_kind': 'full_x',
            'Ay_fit_min': int(FULL_X_LATE_FIT_AY_MIN),
            'Ay_fit_max': int(fit_ay_max),
            'n_fit_points': int(len(fit_sub)),
            'slope_over_ln2': slope,
            'target_slope_over_ln2': float(target),
            'slope_err_over_ln2': float(fit['slope_err']),
            'intercept_over_ln2': intercept,
            'r2': float(fit['r2']),
            'percent_diff': float(percent_diff),
            'fit_coordinate': 'log_sin_pi_Ay_over_Ny',
            'fit_x_min': x_band_min,
            'fit_x_max': x_band_max,
            'plotted_x_min': float(np.min(x_all)),
            'plotted_x_max': float(np.max(x_all)),
            'plotted_Ay_min': int(sub['Ay'].min()),
            'plotted_Ay_max': int(sub['Ay'].max()),
        })

        ax.axvspan(x_band_min, x_band_max, color='0.5', alpha=0.18, label=f'fit Ay={FULL_X_LATE_FIT_AY_MIN}..{fit_ay_max}')
        ax.errorbar(x_all, y_all, yerr=yerr_all, marker='o', ms=3.5, lw=1.3, capsize=2, color='tab:blue', label=r'full_x $S(A_y)/\ln 2$')
        ax.plot(x_line, y_line, color='black', ls='--', lw=1.5, label=f'fit m={slope:.3f}; R²={fit["r2"]:.3f}')
        ax.set_title(f'cycle {PRIMARY_CYCLE}, N16x{int(Ny)}, nshell={int(nshell)}')
        ax.set_xlabel(r'$\log[\sin(\pi A_y/N_y)]$')
        ax.set_ylabel(r'full_x $S(A_y)/\ln 2$')
        outside_legend(ax, fontsize=8)

    fig.suptitle(f'Full-x strip entropy vs log chord, N16x{int(Ny)}; fit window Ay={FULL_X_LATE_FIT_AY_MIN}..Ny//2')
    path = FIG_ROOT / f'full_x_entropy_late_fit_window_N16x{int(Ny)}_cycle{PRIMARY_CYCLE}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(
        path,
        f'Full-x entropy vs log chord, N16x{int(Ny)}',
        'All nonzero Ay values are plotted. The grey band marks the data used for fitting; the dashed line shows that fit extrapolated across the full plotted range.',
    )

full_x_late_fit_rows = pd.DataFrame(full_x_late_fit_rows).sort_values(['Ny', 'nshell']).reset_index(drop=True)
full_x_late_fit_rows.to_csv(OUTPUT_ROOT / 'full_x_late_window_log_chord_fit_rows.csv', index=False)
print('[figures] wrote full_x late-window log-chord fit figures')
display(Markdown('### Full-x late-window log-chord fit rows (`S/ln2`)'))
display(full_x_late_fit_rows)


## 9. Final-Cycle Spatial Contours

Finally, inspect the local half-strip contour at the steady snapshot. These heatmaps are a spatial sanity check for the fitted integrated curves above.

In [ ]:
remove_stale_figures(['final_cycle_half_strip_contour_heatmaps.png'])
for Ny in sorted(run_index['Ny'].unique()):
    indices = [idx for idx, run in enumerate(run_payloads) if int(run['summary']['Ny']) == int(Ny)]
    ncols = len(indices)
    display(Markdown(f'### Half-strip spatial contours: N16x{int(Ny)}'))
    fig, axes = plt.subplots(1, ncols, figsize=(5.2 * ncols, 4.2), constrained_layout=True, sharex=False, sharey=True, squeeze=False)
    axes = axes.ravel()
    vmax = float(np.nanmax([half_contours_cycle50[idx, :, :int(half_Ay[idx])] for idx in indices]))
    last_im = None
    for ax, run_idx in zip(axes, indices):
        run = run_payloads[run_idx]
        summary = run['summary']
        Ay = int(half_Ay[run_idx])
        contour = half_contours_cycle50[run_idx, :, :Ay]
        last_im = ax.imshow(
            contour.T,
            origin='lower',
            aspect='auto',
            cmap='Blues',
            vmin=0.0,
            vmax=vmax,
            extent=[-0.5, int(summary['Nx']) - 0.5, -0.5, Ay - 0.5],
        )
        ax.set_title(f"nsh{summary['nshell']} Ay={Ay}")
        ax.set_xlabel('x')
        ax.set_ylabel('strip dy')
    if last_im is not None:
        fig.colorbar(last_im, ax=axes.tolist(), shrink=0.88, label=r'$s_A(x,dy)/\ln 2$')
    fig.suptitle(f'N16x{int(Ny)} cycle {PRIMARY_CYCLE} half-strip contours')
    path = FIG_ROOT / f'final_cycle_half_strip_contour_heatmaps_N16x{int(Ny)}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_figure(path, f'Half-strip spatial contours, N16x{int(Ny)}, cycle {PRIMARY_CYCLE}')


## 10. Output Manifest

Record the produced tables, arrays, validation report, and figures so this notebook can be treated as a reproducible analysis artifact.

In [ ]:
output_files = sorted(p.name for p in OUTPUT_ROOT.iterdir() if p.is_file())
figure_files = sorted(p.name for p in FIG_ROOT.iterdir() if p.is_file())
expected_outputs = {
    'run_index.csv',
    'strip_entropy_curves.csv',
    'integrated_contour_curves.csv',
    'log_chord_fit_rows.csv',
    'log_chord_fit_summary.csv',
    'full_x_late_window_log_chord_fit_rows.csv',
    'derived_arrays.npz',
    'validation_report.json',
}
for stale_output in ['full_x_log_chord_window_stability_sweep.csv']:
    if stale_output in output_files:
        raise RuntimeError(f'Removed stale output still present: {stale_output}')

missing_outputs = sorted(expected_outputs - set(output_files))
if missing_outputs:
    raise RuntimeError(f'Missing expected outputs: {missing_outputs}')
for stale_name in [
    'strip_entropy_vs_Ay_by_cycle.png',
    'strip_entropy_vs_Ay_cycle50.png',
    'strip_entropy_vs_chord_cycle50.png',
    'log_chord_fits_cycle50.png',
    'slope_percent_error_summary_cycle50.png',
    'full_x_entropy_late_fit_window_cycle50.png',
    'final_cycle_half_strip_contour_heatmaps.png',
    'full_x_entropy_late_fit_window_cycle20.png',
    'full_x_log_chord_window_stability_sweep_cycle50.png',
]:
    if stale_name in figure_files:
        raise RuntimeError(f'Removed mixed-system figure still present: {stale_name}')
if len(figure_files) < 10:
    raise RuntimeError(f'Expected at least 10 figures, found {figure_files}')

analysis_manifest = {
    'analysis_name': 'pure_state_entanglement_vs_system_size_cpu',
    'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
    'output_root': rel_to_bundle(OUTPUT_ROOT),
    'figures_root': rel_to_bundle(FIG_ROOT),
    'primary_cycle': PRIMARY_CYCLE,
    'snapshot_cycles': EXPECTED_SNAPSHOT_CYCLES.astype(int).tolist(),
    'x_intervals': {k: v.astype(int).tolist() for k, v in X_INTERVALS.items()},
    'target_slopes_over_ln2': TARGET_SLOPES,
    'target_slopes_natural_log': TARGET_SLOPES_NATURAL_LOG,
    'entropy_units': ENTROPY_UNITS,
    'system_size_grouping': [system_label(16, ny) for ny in sorted(run_index['Ny'].unique())],
    'run_count': int(len(run_index)),
    'strip_entropy_rows': int(len(strip_entropy_curves)),
    'integrated_contour_rows': int(len(integrated_contour_curves)),
    'fit_rows': int(len(log_chord_fit_rows)),
    'outputs': output_files,
    'figures': figure_files,
    'notes': [
        'Entropy CSV columns, fit slopes, and figures are normalized as S/ln2.',
        'Figures are separated by system size; nshell variants within a system size are grouped as subplots/curves.',
        'Wall-window integrated curves are mean-only because cross-bin covariance is not stored.',
        'Fits use observed Ay=2..Ny//2 unless a later-window diagnostic explicitly states Ay=8..Ny//2.',
    ],
}
write_json_atomic(OUTPUT_ROOT / 'analysis_manifest.json', analysis_manifest)
print(f'[done] outputs written to {OUTPUT_ROOT}')
print(f'[done] figures written to {FIG_ROOT}')
display(Markdown('### Output manifest'))
display(pd.DataFrame({'outputs': output_files + [f'figures/{name}' for name in figure_files]}))
display(Markdown(f'Analysis output folder: `{rel_to_bundle(OUTPUT_ROOT)}`'))
